# Live Demo: Grad-CAM for Image Captioning

This notebook shows how the Grad-CAM idea can be transferred from image classification to **image captioning**.

Instead of explaining a class label, we explain **individual words in a generated caption**.  
The heatmap shows which image regions contributed to a selected word.

In [1]:
import os
import warnings

import cv2
import numpy as np
import matplotlib.pyplot as plt
import torch
from PIL import Image
from transformers import BlipProcessor, BlipForConditionalGeneration

warnings.filterwarnings("ignore")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Load BLIP, a vision-language model for image captioning
model_name = "Salesforce/blip-image-captioning-base"
processor = BlipProcessor.from_pretrained(model_name)
model = BlipForConditionalGeneration.from_pretrained(model_name).to(device)

model.eval()

# Grad-CAM needs gradients, so we explicitly enable them.
for param in model.parameters():
    param.requires_grad = True

print("BLIP model loaded successfully!")

/home/marlon/repos/GRAD_CAM/GRAD_CAM-Presentation/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cpu


Loading weights: 100%|██████████| 473/473 [00:00<00:00, 43743.65it/s]


BLIP model loaded successfully!


## Step 1: Attach hooks to the vision encoder

BLIP first extracts visual features from the image and then generates a caption.  
To create a Grad-CAM map, we save:

- the **activations** of the last vision-encoder layer
- the **gradients** flowing back from the selected word

In [2]:
activations = None
gradients = None

def forward_hook(module, input, output):
    """Store activations from the selected vision layer."""
    global activations
    activations = output[0] if isinstance(output, tuple) else output

def backward_hook(module, grad_input, grad_output):
    """Store gradients flowing back through the selected vision layer."""
    global gradients
    gradients = grad_output[0] if isinstance(grad_output, tuple) else grad_output

# We use the last layer of BLIP's vision encoder as the Grad-CAM target layer.
target_layer = model.vision_model.encoder.layers[-1]
target_layer.register_forward_hook(forward_hook)
target_layer.register_full_backward_hook(backward_hook)

print("Hooks attached to the BLIP vision encoder.")

Hooks attached to the BLIP vision encoder.


## Step 2: Generate a caption and explain one word

The function below:

1. generates a caption for the input image  
2. finds the selected target word in the generated caption  
3. backpropagates from that word's logit score  
4. creates a Grad-CAM heatmap over the image

In [3]:
def generate_blip_gradcam(image_path, target_word):
    # Load and preprocess the image
    original_image = Image.open(image_path).convert("RGB")
    image = original_image.resize((224, 224))
    pixel_values = processor(images=image, return_tensors="pt").pixel_values.to(device)

    # Generate a caption with BLIP
    output_ids = model.generate(pixel_values, max_length=20)
    generated_text = processor.decode(output_ids[0], skip_special_tokens=True)
    print(f"Generated caption: '{generated_text}'")

    # Find the token position that completes the selected target word
    word_index = -1
    target_token_id = -1
    current_string = ""

    for i, token_id in enumerate(output_ids[0]):
        token_str = processor.decode([token_id])
        current_string += token_str

        if target_word.lower() in current_string.lower():
            word_index = i
            target_token_id = token_id
            print(f"Target word '{target_word}' matched at token position {i}.")
            break

    if word_index <= 0:
        print(f"Could not match the word '{target_word}' in the generated caption.")
        print("Tip: choose a word that appears exactly in the generated caption.")
        return image, None, generated_text

    # Backpropagate from the logit score of the selected word.
    # The decoder predicts token i from the previous position i-1.
    model.zero_grad()
    outputs = model(pixel_values=pixel_values, input_ids=output_ids)
    target_score = outputs.logits[0, word_index - 1, target_token_id]
    target_score.backward(retain_graph=True)

    # Grad-CAM computation
    global activations, gradients

    # Remove the CLS token and reshape patch tokens into a 2D grid.
    act = activations[0, 1:, :]
    grad = gradients[0, 1:, :]

    grid_size = int(np.sqrt(act.shape[0]))

    act = act.reshape(grid_size, grid_size, -1).permute(2, 0, 1)
    grad = grad.reshape(grid_size, grid_size, -1).permute(2, 0, 1)

    # Average gradients to obtain importance weights.
    pooled_gradients = torch.mean(grad, dim=[1, 2])

    # Weight each activation map by its corresponding gradient importance.
    for i in range(act.shape[0]):
        act[i, :, :] *= pooled_gradients[i]

    heatmap = torch.sum(act, dim=0).detach().cpu().numpy()

    # Keep only positive evidence for the selected word.
    heatmap = np.maximum(heatmap, 0)

    if np.max(heatmap) > 0:
        heatmap /= np.max(heatmap)

    return image, heatmap, generated_text

## Step 3: Run the image-captioning Grad-CAM demo

Choose an image and a target word from the generated caption.

For the presentation, the important message is:  
**Grad-CAM shows whether a generated word is visually grounded in a meaningful image region.**

In [ ]:
# Choose your own image here.
# The target word should appear in the generated caption.
image_path = "test_images/football2.png"
target_word = "player"

if not os.path.exists(image_path):
    raise FileNotFoundError(
        f"Image not found: {image_path}\n"
        "Please update image_path to a valid local image before running the demo."
    )

img_pil, heatmap, generated_caption = generate_blip_gradcam(image_path, target_word)

if heatmap is not None:
    img_cv = cv2.cvtColor(np.array(img_pil), cv2.COLOR_RGB2BGR)

    heatmap_resized = cv2.resize(heatmap, (img_cv.shape[1], img_cv.shape[0]))
    heatmap_color = cv2.applyColorMap(np.uint8(255 * heatmap_resized), cv2.COLORMAP_JET)

    superimposed_img = heatmap_color * 0.5 + img_cv * 0.5
    superimposed_img = np.clip(superimposed_img, 0, 255).astype(np.uint8)

    plt.figure(figsize=(12, 6))

    plt.subplot(1, 2, 1)
    plt.imshow(img_pil)
    plt.title(f"Generated caption:\n{generated_caption}")
    plt.axis("off")

    plt.subplot(1, 2, 2)
    plt.imshow(cv2.cvtColor(superimposed_img, cv2.COLOR_BGR2RGB))
    plt.title(f"Grad-CAM for word: '{target_word}'")
    plt.axis("off")

    plt.tight_layout()
    plt.show()

FileNotFoundError: Image not found: test_images/fotball2.png
Please update image_path to a valid local image before running the demo.